# Introduction

This Notebook introduces Gemma 4:E2B (it).

Recently Gemma 4 was released. Their capabilities include OCR, speech-to-text, object detection. They also support text-only, multimodal function calling, as well as reasoning, code completion, and correction.

Here are the features of each model:

* Gemma 4 E2B — has a context window of 128K. Parameter size is 2.3B effective, 5.1B with embeddings.
* Gemma 4 E4B — has a context window of 128K. Parameter size is 4.5B effective, 8B with embeddings.
* Gemma 4 26B — mixture of experts with 4B activated/26 total parameters. Context window in 256K.
* Gemma 4 31B — it is a 31B dense model, with context window 256K.
All models have both base and it versions. We will use Ollama on a local machine to run Gemma 4 E2B.



# Upgrade transformers

In [1]:
!pip install -U transformers  -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 73.0 MB/s eta 0:00:00


# Import packages

In [2]:
from time import time
from IPython.display import Markdown
import IPython.display as ipd
from IPython.display import Audio 
from IPython.display import display, HTML
import requests
from PIL import Image
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
import tempfile
import librosa

# Initialize model

In [3]:
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

# Test with text input

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that specializes in answering shortly to any question."},
    {"role": "user", "content": "What is the distance from Earth to the Moon?"},
]

s_time = time()
text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True, 
    enable_thinking=True
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]


outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

processor.parse_response(response)
e_time = time()
total_time = round(e_time - s_time, 2)
print(f"Total time: {total_time}")

Total time: 102.36


Let's display the output more user friendly.

In [5]:
output = processor.parse_response(response)

# Format the output

In [6]:
def colorize_text(text):
    for word, color in zip(["Thinking", "Thinking Process", "Response", "Total time"], ["blue", "red", "green", "magenta"]):
        text = text.replace(f"{word}:", f"\n\n**<font color='{color}'>{word}:</font>**")
    return text
    
def display_response(output, total_time):
    if output.get("thinking"):
        display(Markdown(f"<font color='green'>**Thinking**</font>"))
        display(Markdown(colorize_text(output["thinking"])))
    if output.get("content"):
        display(Markdown(f"<font color='blue'>**Response**</font>"))
        display(Markdown(output["content"]))
    display(Markdown(colorize_text(f"Total time: {total_time} sec.")))

In [7]:
display_response(output, total_time)

<font color='green'>**Thinking**</font>



**<font color='red'>Thinking Process:</font>**

1.  **Analyze the Request:** The user is asking for the distance from Earth to the Moon.
2.  **Determine the Required Information:** I need the approximate average distance between the Earth and the Moon.
3.  **Recall/Verify Knowledge (Internal Knowledge):** The average distance is approximately 384,400 kilometers or 238,900 miles.
4.  **Formulate a Short Answer:** The response needs to be brief, as per the instruction ("answer shortly to any question").
5.  **Final Output Generation.**

<font color='blue'>**Response**</font>

The average distance from Earth to the Moon is about 384,400 kilometers (238,900 miles).



**<font color='magenta'>Total time:</font>** 102.36 sec.

# Test with image

<img src="https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"/>

In [8]:
image_url = "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What you can see in this image?"}
        ]
    }
]
s_time = time()

text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True, 
    enable_thinking=True
)
inputs = processor(
    text=text,
    images=image,
    return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=1024)

response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
output = processor.parse_response(response)

e_time = time()
total_time = round(e_time - s_time, 2)

print(output)
print(f"Total time: {total_time}")

{'role': 'assistant', 'thinking': 'Here\'s a thinking process to generate the image description:\n\n1.  **Analyze the Request:** The user has provided an image and asked, "What you can see in this image?"\n\n2.  **Examine the Image (Initial Scan & Subject Identification):**\n    *   **Main Subject:** A cow (or bovine animal).\n    *   **Appearance of the Cow:** It is brown/reddish-brown, standing, facing slightly toward the camera. It has white markings on its face/head.\n    *   **Setting/Background:**\n        *   Foreground: Sand/beach. Some wetness near the cow\'s feet.\n        *   Middle Ground: The ocean/sea (clear, turquoise/blue water).\n        *   Background: A distant coastline/landmass (hills or islands) under a bright, blue sky.\n    *   **Atmosphere/Lighting:** Bright, sunny, clear day (suggests a warm climate).\n\n3.  **Detail Analysis (Refinement):**\n    *   *The Cow:* It looks healthy. It is the focus.\n    *   *The Environment:* It\'s a tropical or coastal beach sce

Let's display the output nicely formated.

In [9]:
display_response(output, total_time)

<font color='green'>**Thinking**</font>

Here's a thinking process to generate the image description:

1.  **Analyze the Request:** The user has provided an image and asked, "What you can see in this image?"

2.  **Examine the Image (Initial Scan & Subject Identification):**
    *   **Main Subject:** A cow (or bovine animal).
    *   **Appearance of the Cow:** It is brown/reddish-brown, standing, facing slightly toward the camera. It has white markings on its face/head.
    *   **Setting/Background:**
        *   Foreground: Sand/beach. Some wetness near the cow's feet.
        *   Middle Ground: The ocean/sea (clear, turquoise/blue water).
        *   Background: A distant coastline/landmass (hills or islands) under a bright, blue sky.
    *   **Atmosphere/Lighting:** Bright, sunny, clear day (suggests a warm climate).

3.  **Detail Analysis (Refinement):**
    *   *The Cow:* It looks healthy. It is the focus.
    *   *The Environment:* It's a tropical or coastal beach scene. The water is very clear.
    *   *Composition:* The cow dominates the lower half/center. The horizon line separates the beach/water from the sky/landmass.

4.  **Synthesize the Description (Drafting the response):** Start with the main subject, then move to the setting, and finally describe the mood/details.

    *   *(Self-Correction/Enhancement during drafting):* Make sure to use vivid adjectives (e.g., "vibrant," "turquoise," "sunny").

5.  **Final Review and Formatting (The Output):** (This leads to the structured response provided below.) (Check against the original image to ensure accuracy.)

6.  **Final Output Generation.** (The generated response is ready.)

<font color='blue'>**Response**</font>

This image captures a beautiful, sunny coastal scene featuring a cow.

Here is a detailed breakdown of what is visible:

**Main Subject:**
*   **A Cow:** The central focus of the image is a bovine animal, likely a cow or steer. It has a rich reddish-brown coat and distinct white markings on its face and head. It is standing on the beach, looking toward the camera.

**Setting and Environment:**
*   **Beach:** The foreground consists of light-colored sand. There appears to be some wetness or tidal area near the cow's feet.
*   **Ocean/Water:** In the middle ground, there is clear, turquoise or light blue ocean water.
*   **Background:** In the distance, there is a strip of land or coastline, possibly featuring hills or islands, meeting the horizon.
*   **Sky:** The sky is bright blue with a few wispy white clouds, indicating a very sunny day.

**Overall Impression:**
The image conveys a warm, idyllic, and peaceful vacation or rural coastal atmosphere. The bright lighting and vibrant colors of the water and sky make for a very attractive scene.



**<font color='magenta'>Total time:</font>** 595.69 sec.

# Test with sound


In [10]:
def cstr(str_text, color='black'):
    """
    Html styling for widgets
    Args
        str_text: text to disply
        color: color to display the text
    Returns
        Formated text/label
    """
    return "<text style=color:{}><strong>{}<strong></text>".format(color, str_text)

def play_sound(sound_path="",
               text="Test", 
               color="green"):
    """
    Display a sound play widget
    Args
        sound_path: path to the sound file
        text: text to display
        color: color for text to display
    Returns
        None
    """
    display(HTML(cstr(text, color)))
    display(ipd.Audio(sound_path))

In [11]:
def download_audio_data(url):
    """
    Download audio data
    Args
        url: url for the audio data
    Returns
        the name of the local saved audio file
    """
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
        return f.name

In [12]:
audio_url = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/journal1.wav"
audio_path = download_audio_data(audio_url)

In [13]:
play_sound(audio_path, text="Journal", color="blue")

In [14]:
audio_array, sr = librosa.load(audio_path, sr=16000)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": audio_array},
            {"type": "text", "text": "Transcribe the following audio exactly. Only output transcription."}
        ]
    }
]
s_time = time()

text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True, 
    enable_thinking=True
)
inputs = processor(
    text=text,
    audio=audio_array,
    return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=128)

response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
output = processor.parse_response(response)

e_time = time()
total_time = round(e_time - s_time, 2)

print(output)
print(f"Total time: {total_time}")

{'role': 'assistant', 'content': '<|channel>thought\nThinking Process:\n\n1.  **Analyze the Request:** The user wants me to transcribe the provided audio exactly.\n2.  **Analyze the Audio (Input):** "okaboali today feelingly fresh the morning light was beautiful and i enjoyed a nice kabo coffee"\n3.  **Perform Transcription & Verification (Aural/Textual comparison):** I need to listen/read the audio and ensure accuracy, paying attention to sounds and phrasing.\n\n    *   *Input audio:* "okaboali today feelingly fresh the morning light was beautiful and i enjoyed a nice kabo coffee"\n\n'}
Total time: 134.88


In [15]:
display_response(output, total_time)

<font color='blue'>**Response**</font>

<|channel>thought
Thinking Process:

1.  **Analyze the Request:** The user wants me to transcribe the provided audio exactly.
2.  **Analyze the Audio (Input):** "okaboali today feelingly fresh the morning light was beautiful and i enjoyed a nice kabo coffee"
3.  **Perform Transcription & Verification (Aural/Textual comparison):** I need to listen/read the audio and ensure accuracy, paying attention to sounds and phrasing.

    *   *Input audio:* "okaboali today feelingly fresh the morning light was beautiful and i enjoyed a nice kabo coffee"





**<font color='magenta'>Total time:</font>** 134.88 sec.

# Preliminary conclusions


We tested Gemma 4:E2B on CPU for text, image, andsound. Although slow compared with GPU (tens of second for text only compared with less than a second on GPU, and several minutes for an image interpretation), the response is good quality.

We will follow with a version using GPU to compare the performance from execution time perspective.
